# Deep learning model: Long Short Term Memory (LSTM)
### Implemented by: Junior Chaj Mejia

### Collect packages

In [ ]:
from collections import Counter
from datasets import load_dataset
from sklearn.metrics import accuracy_score, f1_score, precision_score, recall_score
from torch.utils.data import Dataset, DataLoader
import numpy as np
import os
import pandas as pd
import pandas as pd
import re
import torch
import torch.nn as nn

### Pre-process for LSTM

In [6]:
print("Loading dataset from HuggingFace...")
emotion_dataset = load_dataset("dair-ai/emotion")

train_data = pd.DataFrame(emotion_dataset["train"])
test_data  = pd.DataFrame(emotion_dataset["test"])
val_data   = pd.DataFrame(emotion_dataset["validation"])

print(f"Train: {len(train_data)} | Test: {len(test_data)} | Val: {len(val_data)}")

def clean_text_lstm(text):
    text = text.lower()
    text = re.sub(r"http\S+|www\S+", "", text)
    text = re.sub(r"@\w+", "", text)
    text = re.sub(r"#(\w+)", r"\1", text)
    text = re.sub(r"\b(not|no|never|n't)\s+(\w+)", r"\1_\2", text)
    text = re.sub(r"[^a-z!?_\s]", "", text)
    text = re.sub(r"!+", "!", text)
    text = re.sub(r"\?+", "?", text)
    text = re.sub(r"\s+", " ", text).strip()
    return text

train_data["cleaned_text"] = train_data["text"].apply(clean_text_lstm)
test_data["cleaned_text"]  = test_data["text"].apply(clean_text_lstm)
val_data["cleaned_text"]   = val_data["text"].apply(clean_text_lstm)

rows_before = len(train_data)
train_data = train_data.drop_duplicates(subset="text").reset_index(drop=True)
print(f"Duplicates removed from train: {rows_before - len(train_data)}")

train_df = train_data
val_df   = val_data
test_df  = test_data

Loading dataset from HuggingFace...
Train: 16000 | Test: 2000 | Val: 2000
Duplicates removed from train: 31


In [7]:
train_data["seq_len"] = train_data["cleaned_text"].apply(lambda x: len(x.split()))
print("\nSequence length stats (train):")
print(train_data["seq_len"].describe())

# Inspect samples
print(train_data["cleaned_text"].head(10))


Sequence length stats (train):
count    15969.000000
mean        19.015280
std         10.896035
min          1.000000
25%         11.000000
50%         17.000000
75%         25.000000
max         64.000000
Name: seq_len, dtype: float64
0                              i didnt feel humiliated
1    i can go from feeling so hopeless to so damned...
2     im grabbing a minute to post i feel greedy wrong
3    i am ever feeling nostalgic about the fireplac...
4                                 i am feeling grouchy
5    ive been feeling a little burdened lately wasn...
6    ive been taking or milligrams or times recomme...
7    i feel as confused about life as a teenager or...
8    i have been with petronas for years i feel tha...
9                                  i feel romantic too
Name: cleaned_text, dtype: str


### Functions for dataset

In [12]:
def build_vocab(texts, max_vocab_size=20000, min_freq=2):
    counter = Counter()
    for text in texts:
        counter.update(text.split())
    vocab = {'<PAD>': 0, '<UNK>': 1}
    for word, freq in counter.most_common(max_vocab_size):
        if freq >= min_freq:
            vocab[word] = len(vocab)
    return vocab

vocab = build_vocab(train_df['cleaned_text'])
vocab_size = len(vocab)
print(f"Vocabulary size: {vocab_size}")

MAX_LEN = 60  # slightly longer than original 50

def encode(text, vocab, max_len=MAX_LEN):
    tokens = text.split()
    ids = [vocab.get(t, vocab['<UNK>']) for t in tokens][:max_len]
    ids += [vocab['<PAD>']] * (max_len - len(ids))
    return ids

Vocabulary size: 7598


### Configure Dataset

In [13]:
label_counts = Counter(train_df['label'])
num_classes  = 6
total        = sum(label_counts.values())
class_weights = torch.tensor(
    [total / (num_classes * label_counts[i]) for i in range(num_classes)],
    dtype=torch.float
)
print("Class weights:", class_weights.tolist())

class EmotionDataset(Dataset):
    def __init__(self, df, vocab, max_len=MAX_LEN):
        self.texts  = df['cleaned_text'].tolist()
        self.labels = df['label'].tolist()
        self.vocab  = vocab
        self.max_len = max_len

    def __len__(self):
        return len(self.texts)

    def __getitem__(self, idx):
        x = torch.tensor(encode(self.texts[idx], self.vocab, self.max_len), dtype=torch.long)
        y = torch.tensor(self.labels[idx], dtype=torch.long)
        return x, y


Class weights: [0.5706475377082825, 0.49747663736343384, 2.0488836765289307, 1.2350348234176636, 1.3768752813339233, 4.685739517211914]


### LSTM Model

In [ ]:
class LSTMClassifier(nn.Module):
    def __init__(self, vocab_size, embed_dim=128, hidden_dim=256,
                 num_classes=6, num_layers=2, dropout=0.4):
        super().__init__()
        self.embedding = nn.Embedding(vocab_size, embed_dim, padding_idx=0)
        self.lstm = nn.LSTM(
            embed_dim, hidden_dim,
            num_layers=num_layers,
            batch_first=True,
            bidirectional=True,   # ← reads text both ways
            dropout=dropout if num_layers > 1 else 0.0
        )
        self.fc      = nn.Linear(hidden_dim * 2, num_classes)
        self.dropout = nn.Dropout(dropout)

    def forward(self, x):
        x = self.dropout(self.embedding(x))
        _, (hidden, _) = self.lstm(x)
        # Concat final forward & backward hidden states from last layer
        out = torch.cat([hidden[-2], hidden[-1]], dim=1)
        out = self.dropout(out)
        return self.fc(out)

### Training configuration

In [15]:
device = torch.device('mps' if torch.cuda.is_available() else 'cpu')
print(f"Using device: {device}")

train_loader = DataLoader(EmotionDataset(train_df, vocab), batch_size=64, shuffle=True)
val_loader   = DataLoader(EmotionDataset(val_df,   vocab), batch_size=64)
test_loader  = DataLoader(EmotionDataset(test_df,  vocab), batch_size=64)

model     = LSTMClassifier(vocab_size).to(device)
optimizer = torch.optim.Adam(model.parameters(), lr=3e-4, weight_decay=1e-5)
criterion = nn.CrossEntropyLoss(weight=class_weights.to(device))  # weighted loss

# Reduce LR when val F1 plateaus
scheduler = torch.optim.lr_scheduler.ReduceLROnPlateau(
    optimizer, mode='max', factor=0.5, patience=2,
)

Using device: cpu


### Evaluation configuration

In [16]:
def evaluate(loader):
    model.eval()
    preds, labels_all = [], []
    with torch.no_grad():
        for x, y in loader:
            x, y = x.to(device), y.to(device)
            logits = model(x)
            pred = torch.argmax(logits, dim=1)
            preds.extend(pred.cpu().numpy())
            labels_all.extend(y.cpu().numpy())
    return {
        'accuracy':         accuracy_score(labels_all, preds),
        'precision_macro':  precision_score(labels_all, preds, average='macro', zero_division=0),
        'recall_macro':     recall_score(labels_all, preds, average='macro', zero_division=0),
        'f1_macro':         f1_score(labels_all, preds, average='macro', zero_division=0),
    }

def fmt(m):
    return (f"Acc={m['accuracy']:.4f}  Prec={m['precision_macro']:.4f}  "
            f"Rec={m['recall_macro']:.4f}  F1={m['f1_macro']:.4f}")

### Training loop

In [17]:
epochs       = 20
best_val_f1  = 0.0
patience     = 4          # early stopping patience
no_improve   = 0
best_path    = "best_lstm.pt"

for epoch in range(epochs):
    model.train()
    total_loss = 0

    for x, y in train_loader:
        x, y = x.to(device), y.to(device)
        optimizer.zero_grad()
        logits = model(x)
        loss   = criterion(logits, y)
        loss.backward()
        nn.utils.clip_grad_norm_(model.parameters(), max_norm=1.0)  # gradient clipping
        optimizer.step()
        total_loss += loss.item()

    val_metrics = evaluate(val_loader)
    avg_loss    = total_loss / len(train_loader)
    val_f1      = val_metrics['f1_macro']

    print(f"Epoch {epoch+1:02d} | Loss: {avg_loss:.4f} | Val: {fmt(val_metrics)}")

    scheduler.step(val_f1)
    current_lr = optimizer.param_groups[0]['lr']
    print(f"  LR: {current_lr:.2e}")

    # Save best checkpoint
    if val_f1 > best_val_f1:
        best_val_f1 = val_f1
        no_improve  = 0
        torch.save(model.state_dict(), best_path)
        print(f"  ✓ New best val F1: {best_val_f1:.4f} — checkpoint saved")
    else:
        no_improve += 1
        if no_improve >= patience:
            print(f"  Early stopping at epoch {epoch+1}")
            break

Epoch 01 | Loss: 1.7791 | Val: Acc=0.2785  Prec=0.2125  Rec=0.2508  F1=0.1904
  LR: 3.00e-04
  ✓ New best val F1: 0.1904 — checkpoint saved
Epoch 02 | Loss: 1.6545 | Val: Acc=0.3450  Prec=0.3674  Rec=0.3901  F1=0.3517
  LR: 3.00e-04
  ✓ New best val F1: 0.3517 — checkpoint saved
Epoch 03 | Loss: 1.4781 | Val: Acc=0.4460  Prec=0.4732  Rec=0.5119  F1=0.4747
  LR: 3.00e-04
  ✓ New best val F1: 0.4747 — checkpoint saved
Epoch 04 | Loss: 1.3046 | Val: Acc=0.5160  Prec=0.5329  Rec=0.6087  F1=0.5448
  LR: 3.00e-04
  ✓ New best val F1: 0.5448 — checkpoint saved
Epoch 05 | Loss: 1.1592 | Val: Acc=0.5965  Prec=0.6154  Rec=0.6779  F1=0.6142
  LR: 3.00e-04
  ✓ New best val F1: 0.6142 — checkpoint saved
Epoch 06 | Loss: 1.0176 | Val: Acc=0.6585  Prec=0.6559  Rec=0.7283  F1=0.6738
  LR: 3.00e-04
  ✓ New best val F1: 0.6738 — checkpoint saved
Epoch 07 | Loss: 0.8913 | Val: Acc=0.7355  Prec=0.7151  Rec=0.7852  F1=0.7431
  LR: 3.00e-04
  ✓ New best val F1: 0.7431 — checkpoint saved
Epoch 08 | Loss: 0.7

### Evaluation on testing

In [ ]:
print("\nLoading best checkpoint for test evaluation...")
model.load_state_dict(torch.load(best_path))

test_metrics = evaluate(test_loader)
print(f"\nTest results: {fmt(test_metrics)}")


Loading best checkpoint for test evaluation...

Test results: Acc=0.8950  Prec=0.8417  Rec=0.8995  F1=0.8641


### View sample predictions

In [19]:
LABEL_NAMES = {0: 'sadness', 1: 'joy', 2: 'love', 3: 'anger', 4: 'fear', 5: 'surprise'}

def show_samples(loader, df, n=5):
    model.eval()
    all_preds, all_labels, all_texts = [], [], []

    with torch.no_grad():
        for x, y in loader:
            x, y = x.to(device), y.to(device)
            preds = torch.argmax(model(x), dim=1)
            all_preds.extend(preds.cpu().numpy())
            all_labels.extend(y.cpu().numpy())

    # Uncleaned for readability
    all_texts = df['text'].tolist()  

    results = pd.DataFrame({
        'text':      all_texts[:len(all_preds)],
        'actual':    [LABEL_NAMES[l] for l in all_labels],
        'predicted': [LABEL_NAMES[p] for p in all_preds],
    })
    results['correct'] = results['actual'] == results['predicted']

    print("=" * 70)
    print(f"CORRECTLY CLASSIFIED (n={n})")
    print("=" * 70)
    correct = results[results['correct']].sample(n=n, random_state=42)
    for _, row in correct.iterrows():
        print(f"  [{row['actual']}]  {row['text']}")

    print()
    print("=" * 70)
    print(f"MISCLASSIFIED (n={n})")
    print("=" * 70)
    wrong = results[~results['correct']].sample(n=n, random_state=42)
    for _, row in wrong.iterrows():
        print(f"  actual={row['actual']}  predicted={row['predicted']}")
        print(f"  {row['text']}")
        print()

show_samples(test_loader, test_df, n=5)

CORRECTLY CLASSIFIED (n=5)
  [joy]  i like to think i present myself and the life and times of the working mum to a good standard and if i ever do miss a apostrophe or miss spell a particular word please feel free to call me on it
  [sadness]  i suppose we all feel a little inhibited when it comes to picking up the phone and calling someone we re not very close to anymore
  [joy]  i feel a little virtuous doing these things but on the other hand nini s tasted better
  [sadness]  i feel ugly and hated
  [love]  i am definitely feeling the effects of the progesterone in two ways my breasts are tender and i m tired

MISCLASSIFIED (n=5)
  actual=anger  predicted=fear
  i took for granted a few weeks ago is really weird and makes me feel really agitated and frustrated

  actual=joy  predicted=love
  i firmly believe that the only way to go about this craft is to write the book that you feel passionate about and not to worry about finding the book that the mass audience desires

  actual=joy